### Dependencias, Ingesta y Copia de Seguridad

In [5]:
import pandas as pd
import numpy as np
import unicodedata

# 1. Cargar el archivo crudo respetando su delimitador y codificación de origen
df_raw = pd.read_csv('../data/raw/oferta_gastronomica.csv', sep=';', encoding='latin1')

# 2. Crear una copia de trabajo independiente para no alterar nunca los datos originales
df = df_raw.copy()

print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")


Dataset cargado: 2823 filas x 18 columnas


### Diagnostico de datos

In [6]:
diagnostico = pd.DataFrame({
    'Tipo_Dato': df.dtypes,
    'Nulos': df.isnull().sum(),
    'Porcentaje_Nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'Valores_Unicos': df.nunique()
})

diagnostico


,Tipo_Dato,Nulos,Porcentaje_Nulos,Valores_Unicos
long,str,0,0.00,2703
lat,str,0,0.00,2702
id,int64,0,0.00,2823
nombre,str,0,0.00,2554
categoria,str,116,4.11,8
cocina,str,359,12.72,52
ambientacion,str,559,19.80,57
telefono,str,520,18.42,2246
mail,str,2351,83.28,403
horario,str,2098,74.32,387


### Reparación Global de Codificación (Mojibake)

In [7]:
#Calculo la cantidad de filas duplcadas
duplicados = df.duplicated().sum()
print(f'Cantidad de filas duplicadas: {duplicados}')

Cantidad de filas duplicadas: 0


In [8]:
#Comienzo con la estandarizacion y limpieza
#realizo copia de seguridad
df_limpio = df.copy()
print(df_limpio.isnull().sum())


long                          0
lat                           0
id                            0
nombre                        0
categoria                   116
cocina                      359
ambientacion                559
telefono                    520
mail                       2351
horario                    2098
calle_nombre                  0
calle_altura                  0
calle_cruce                2817
direccion_completa            1
barrio                        0
comuna                        0
codigo_postal                57
codigo_postal_argentino      66
dtype: int64


In [10]:
#trato valores nulos y estandarizo colmnas de texto
# Relleno los NaN con 'No especificado' para no perder registros 
columnas_texto = ['categoria', 'cocina', 'ambientacion', 'barrio']
for col in columnas_texto:
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio[col].fillna(f'{col} no especificado')
        # Estandarizamos el texto: pasa a string, se quitan espacios extra y se aplico formato título
        df_limpio[col] = df_limpio[col].astype(str).str.strip().str.title()

In [11]:
#agrupacion por comuna y barrio. comuna (visión macro y logística) y barrio (detalle micro local 
# para detectar baches de oferta y saturación comercial con precisión)
df_comuna = df_limpio.groupby(['comuna', 'barrio']).agg(
    total_establecimientos=('nombre', 'count'),
    categorias_unicas=('categoria', 'nunique')
).reset_index()

In [12]:
#ordenado de mayor a menor 
df_comuna = df_comuna.sort_values(by='total_establecimientos', ascending=False)

In [13]:
print('Comunas y barrios con mayor densidad: ')
print(df_comuna.head())

Comunas y barrios con mayor densidad: 
    comuna       barrio  total_establecimientos  categorias_unicas
41      14      Palermo                     467                  9
4        1  San Nicolas                     454                  7
6        2     Recoleta                     327                  8
7        3    Balvanera                     215                  8
1        1    Monserrat                     149                  5


In [14]:
# Exportar los datasets limpios a la carpeta data/processed/
df_limpio.to_csv("../data/processed/oferta_gastronomica_limpia.csv", index=False, encoding="utf-8")
df_comuna.to_csv("../data/processed/densidad_comercial_por_comuna.csv", index=False, encoding="utf-8")


In [16]:
df_limpio = df.copy()
df_limpio.to_csv("../data/processed/oferta_gastronomica_limpia.csv", index=False, encoding="utf-8")
df_limpio = pd.read_csv('../data/processed/oferta_gastronomica_limpia.csv',  encoding="utf-8")

def corregir_encoding(texto):
    if isinstance(texto, str):
        try:
            return texto.encode("latin-1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return texto
    return texto

cols_objeto = df.select_dtypes(include=['object', 'string']).columns.tolist()
df[cols_objeto] = df[cols_objeto].map(corregir_encoding)


### Estandarización de Textos y Normalización Unicode

In [17]:
def remover_tildes(valor):
    if pd.isna(valor) or not isinstance(valor, str):
        return valor
    return ''.join(
        c for c in unicodedata.normalize('NFD', valor)
        if unicodedata.category(c) != 'Mn'
    )

# 1. Limpieza general en todas las columnas de texto (excepto 'mail' y 'horario' que se procesan aparte)
cols_limpieza_general = [c for c in cols_objeto if c not in ['mail', 'horario']]

df = df_limpio.map(corregir_encoding)

# 2. Quitar tildes en columnas de agrupación para evitar fragmentar datos
cols_sin_tildes = ['barrio', 'categoria', 'cocina', 'ambientacion']
for col in cols_sin_tildes:
    df[col] = df[col].apply(remover_tildes)


### Coordenadas Geográficas, Imputación de Nulos y Descarte

In [24]:
# 1. Convertir coordenadas de texto con coma a números decimales (float)
for coord in ['long', 'lat']:
    df[coord] = (
        df[coord]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .replace({'nan': np.nan, 'NAN': np.nan})
        .astype(float)
    )

# 2. Imputar nulos en columnas categóricas clave para no perder registros al agrupar
for col in ['categoria', 'cocina', 'ambientacion']:
    df[col] = df[col].fillna('SIN ESPECIFICAR')

# 3. Eliminar duplicados por ID antes de podar columnas
df = df.drop_duplicates(subset=['id']).reset_index(drop=True)

# 4. Eliminacion de columnas con nulos masivos o sin valor analítico para el Punto 2 y cruce con Trends
columnas_a_eliminar = [
    'calle_cruce',              # 99.79% nulos (sin valor)
    'telefono',                 # Dato operativo de contacto, irrelevante para tendencias
    'codigo_postal_argentino',  # Redundante frente a barrio, comuna y codigo_postal
    'calle_nombre',             # Redundante frente a direccion_completa
    'calle_altura'              # Redundante frente a direccion_completa
]

df = df.drop(columns=columnas_a_eliminar, errors='ignore')

In [25]:
# 1. Auditoría de caracteres especiales residuales
for col in df.select_dtypes(include=['object', 'string']).columns:
    serie_valida = df[col].dropna()
    especiales = serie_valida[serie_valida.str.contains(r'[^\x00-\x7F]', regex=True)]
    if len(especiales) > 0:
        print(f"Columna [{col}]: {len(especiales)} filas con caracteres especiales -> {especiales.unique()[:3]}")

# 2. Exportar dataset maestro limpio en data/processed/ (Checkpoint Clase 13)
ruta_salida = '../data/processed/oferta_gastronomica_limpia.csv'
df.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print(f"\nExportación exitosa en '{ruta_salida}': {df.shape[0]} filas x {df.shape[1]} columnas.")

Columna [nombre]: 69 filas con caracteres especiales -> <StringArray>
['A CORUÃ‘A', 'AÃ‘ARO CERO', 'ANTAÃ‘O BAR']
Length: 3, dtype: str
Columna [mail]: 6 filas con caracteres especiales -> <StringArray>
[   'www.lascioletas-cafâ€š.com.ar', 'meetingplace_cafâ€š@yahoo.com.ar',
           'www.moliere_cafâ€š.com']
Length: 3, dtype: str
Columna [horario]: 222 filas con caracteres especiales -> <StringArray>
[                     'lunes a sábado de 8 a 00',
         'lunes a sábado de 9 a 16 y de 19 a 21',
 'lunes a viernes de 12 a 00 - sábado de 20 a 3']
Length: 3, dtype: str
Columna [direccion_completa]: 38 filas con caracteres especiales -> <StringArray>
['HUMBERTO 1º 1843', 'RODRIGUEZ PEÃ‘A 687', 'CERVIÃ‘O AV. 3496']
Length: 3, dtype: str
Columna [barrio]: 28 filas con caracteres especiales -> <StringArray>
['NuA±ez']
Length: 1, dtype: str

Exportación exitosa en '../data/processed/oferta_gastronomica_limpia.csv': 2823 filas x 13 columnas.


### Verificacion de correcta limpieza de los datos

In [ ]:
#exploro para verificar errores, si hay
df_l = pd.read_csv('../data/processed/oferta_gastronomica_limpia.csv')
diagnostico = pd.DataFrame({
    'Tipo_Dato': df_l.dtypes,
    'Nulos': df_l.isnull().sum(),
    'Porcentaje_Nulos': (df_l.isnull().sum() / len(df_l) * 100).round(2),
    'Valores_Unicos': df_l.nunique()
})
print(diagnostico)
print(f'Porcentaje de mails nulos: {diagnostico['Porcentaje_Nulos'].mail}%')
print(f'Porcentaje de horarios nulos: {diagnostico['Porcentaje_Nulos'].horario}%')
print(f'Tipo de dato de los codigos postales: {diagnostico['Tipo_Dato'].codigo_postal}')
#1- casi 3/4 partes de los registros no tienen horario y mas de 3/4 no tienen mail registrados
#2- los codigos postales son float cuando en argentina, no deberia ser asi
#en el primer caso, se deberia eliminar estas columnas y en el segundo se deberia cambiar el 
#tipo de dato a str


                   Tipo_Dato  Nulos  Porcentaje_Nulos  Valores_Unicos
long                 float64      0              0.00            2703
lat                  float64      0              0.00            2702
id                     int64      0              0.00            2823
nombre                   str      0              0.00            2554
categoria                str      0              0.00               9
cocina                   str      0              0.00              53
ambientacion             str      0              0.00              58
mail                     str   2351             83.28             403
horario                  str   2098             74.32             387
direccion_completa       str      1              0.04            2745
barrio                   str      0              0.00              47
comuna                 int64      0              0.00              15
codigo_postal        float64     57              2.02             234
Porcentaje de mails 

### Correcion de los datos erroneos ubicados en la anterior verificacion

In [51]:
#borro las columnas
df_l = df_l.drop(columns=['mail', 'horario'], errors='ignore') 
#cambio el tipo de dato de codigo postal
df_l['codigo_postal'] = df_l['codigo_postal'].astype(str)
df_l['codigo_postal'] = df_l['codigo_postal'].str.replace(r'\.0$', '', regex=True).str.replace(r'\D', '', regex=True).replace('', 'Codigo postal no registrado')
df_l.head() #algunos codigos postales figuran como NaN



,long,lat,id,nombre,categoria,cocina,ambientacion,direccion_completa,barrio,comuna,codigo_postal
0,-58.363307,-34.622963,1602,GUANTANAMERA,RESTAURANTE,CUBANA,MODERNO,"MOREAU DE JUSTO, ALICIA AV. 2092",Puerto Madero,1,1107
1,-58.392330,-34.621554,676,1843 VINACOTECA,VINERIA,SIN ESPECIFICAR,SIN ESPECIFICAR,HUMBERTO 1º 1843,San Cristobal,3,NaN
2,-58.453542,-34.560311,487,2080 EXPRESS,RESTAURANTE,TRADICIONAL,CLASICO PORTENO,JURAMENTO AV. 2080,Belgrano,13,1428
3,-58.475491,-34.677051,2737,22,CAFE,MINUTAS,MODERNO,MURGUIONDO 4188,Villa Lugano,8,1439
4,-58.425437,-34.585695,2,4 EN 2 I,RESTAURANTE,SIN ESPECIFICAR,SIN ESPECIFICAR,"BORGES, JORGE LUIS 2095",Palermo,14,1425
